In [7]:
!pip install transformers seqeval -q

import numpy as np
import torch
import nltk
from nltk.corpus import treebank
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import accuracy_score, f1_score, classification_report
import random

nltk.download('treebank', quiet=True)

# -------------------------
# CONFIG
# -------------------------

MODEL_NAME = "prajjwal1/bert-mini"
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "/kaggle/working/bert_pos_output"

# -------------------------
# BUILD DATASET FROM NLTK TREEBANK
# -------------------------

sentences = list(treebank.tagged_sents())
random.shuffle(sentences)

all_tags = sorted(set(tag for sent in sentences for _, tag in sent))
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)

print(f"Total sentences: {len(sentences)}")
print(f"Number of POS labels: {num_labels}")

def sentences_to_dict(sents):
    return {
        "tokens": [[w for w, _ in s] for s in sents],
        "pos_tags": [[label2id[t] for _, t in s] for s in sents],
    }

train_sents = sentences[:int(0.8 * len(sentences))]
val_sents   = sentences[int(0.8 * len(sentences)):int(0.9 * len(sentences))]
test_sents  = sentences[int(0.9 * len(sentences)):]

dataset = DatasetDict({
    "train":      Dataset.from_dict(sentences_to_dict(train_sents)),
    "validation": Dataset.from_dict(sentences_to_dict(val_sents)),
    "test":       Dataset.from_dict(sentences_to_dict(test_sents)),
})

print(dataset)

# -------------------------
# TOKENISE & ALIGN LABELS
# -------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenise_and_align(examples):
    tokenised = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LEN,
        is_split_into_words=True,
    )
    all_labels = []
    for i, pos_tags in enumerate(examples["pos_tags"]):
        word_ids = tokenised.word_ids(batch_index=i)
        labels = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)
            elif word_id != prev_word_id:
                labels.append(pos_tags[word_id])
            else:
                labels.append(-100)
            prev_word_id = word_id
        all_labels.append(labels)
    tokenised["labels"] = all_labels
    return tokenised

tokenised_dataset = dataset.map(
    tokenise_and_align,
    batched=True,
    remove_columns=["tokens", "pos_tags"],
)

# -------------------------
# MODEL
# -------------------------

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# -------------------------
# METRICS
# -------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [id2label[l] for l in label_seq if l != -100]
        for label_seq in labels
    ]
    pred_labels = [
        [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "f1_macro": f1_score(true_labels, pred_labels, average="macro"),
    }

# -------------------------
# TRAINING
# -------------------------

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_dataset["train"],
    eval_dataset=tokenised_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# -------------------------
# EVALUATE ON TEST SET
# -------------------------

predictions, labels, _ = trainer.predict(tokenised_dataset["test"])
predictions = np.argmax(predictions, axis=-1)

true_labels = [
    [id2label[l] for l in label_seq if l != -100]
    for label_seq in labels
]
pred_labels = [
    [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
    for pred_seq, label_seq in zip(predictions, labels)
]

print("\n--- seqeval Report ---")
print(classification_report(true_labels, pred_labels))
print(f"Token-level Accuracy : {accuracy_score(true_labels, pred_labels):.4f}")
print(f"Macro F1             : {f1_score(true_labels, pred_labels, average='macro'):.4f}")

# -------------------------
# SAMPLE PREDICTION
# -------------------------

def predict(sentence: str):
    words = sentence.split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
    )
    word_ids = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    ).word_ids()
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=-1)[0].tolist()
    seen = set()
    result = []
    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)
        result.append((words[word_id], id2label[preds[idx]]))
    return result

print("\nSample prediction:")
print(predict("The quick brown fox is running quickly"))

Total sentences: 3914
Number of POS labels: 46
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags'],
        num_rows: 3131
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags'],
        num_rows: 391
    })
    test: Dataset({
        features: ['tokens', 'pos_tags'],
        num_rows: 392
    })
})


Map:   0%|          | 0/3131 [00:00<?, ? examples/s]

Map:   0%|          | 0/391 [00:00<?, ? examples/s]

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/69 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNS seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VBD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: RB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VBN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/us

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias'].
There were unexpected keys in the checkp

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VBZ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWar


--- seqeval Report ---
              precision    recall  f1-score   support

           '       1.00      1.00      1.00        67
           B       0.93      0.91      0.92       497
          BD       0.97      0.99      0.98       300
          BG       0.90      0.88      0.89       140
          BN       0.95      0.96      0.95       218
          BP       0.93      0.96      0.95       135
          BR       0.00      0.00      0.00        10
          BZ       1.00      0.97      0.99       207
           C       0.97      0.93      0.95       209
           D       0.99      0.97      0.98       366
          DT       1.00      0.91      0.95        53
           J       0.91      0.86      0.88       552
          JR       0.81      0.95      0.87        40
          JS       1.00      0.95      0.97        19
           N       0.93      0.94      0.93      1691
          NP       0.86      0.89      0.87       553
         NPS       0.00      0.00      0.00        26
   